# CNN-LSTM Maneuver Classifier

This notebook uses only the ego planar lidar sequence. It is a good ablation to show how much graph context helps later.


## Run Safety

Before running this notebook, restart the kernel.

Why this matters:
- previous model runs can leave tensors and cached arrays in memory
- old variables can leak into the current run and make comparisons unfair
- we want every notebook to save a clean, reproducible result

Recommended workflow:
1. Restart kernel
2. Run all cells from top to bottom
3. Let the notebook save its outputs before closing it


In [1]:
# Memory cleanup before starting this notebook.
import gc

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

print('Kernel memory cleanup complete. Start the notebook from here after a restart.')


Kernel memory cleanup complete. Start the notebook from here after a restart.


In [2]:
# Core imports used throughout this notebook.
from dataclasses import dataclass
from pathlib import Path
from collections import Counter
from typing import Optional

import json
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

# Resolve the project structure relative to this notebook.
PROJECT_ROOT = Path.cwd().resolve().parent
EVAL_ROOT = PROJECT_ROOT
RESULTS_ROOT = EVAL_ROOT / 'results'
WEIGHTS_ROOT = EVAL_ROOT / 'model_weights'
SPLITS_ROOT = EVAL_ROOT / 'splits'

ORIGINAL_DATASET_ROOT = PROJECT_ROOT.parent / '03_dataset' / 'husky_control_dataset'
PREFERRED_DATASET_ROOT = ORIGINAL_DATASET_ROOT
DATASET_ROOT = PREFERRED_DATASET_ROOT

print('Original dataset root:', ORIGINAL_DATASET_ROOT)
print('Preferred dataset root:', PREFERRED_DATASET_ROOT)
print('Dataset root:', DATASET_ROOT)


Original dataset root: /home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset
Preferred dataset root: /home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset
Dataset root: /home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset


In [3]:
# Shared experiment configuration.
# Keep this block explicit and heavily commented because it is the main place
# you will tune once more data has been collected.
LABEL_MODE = 'reduced'      # Use the balanced 5-class setting by default.
SEED = 42                   # Fix the split and initialization for reproducibility.
PAST_LEN = 10               # Number of past timesteps used to predict the current maneuver.
FUTURE_LEN = 5             # Number of future steps used for trajectory targets (ADE/FDE/RMSE).
SCAN_BEAMS = 512            # Number of lidar beams after resampling.
RANGE_CLIP = 30.0           # Maximum lidar range used for normalization.
BATCH_SIZE = 32             # Batch size for trainable models.
EPOCHS = 30                 # Upper bound; early stopping will usually stop sooner.
EARLY_STOPPING_PATIENCE = 5 # Patience on validation macro-F1.
LR = 1e-3                   # Adam learning rate.
WEIGHT_DECAY = 1e-4         # Small regularization on trainable models.
TRAJ_LOSS_WEIGHT = 1.0     # Weight for future-trajectory regression alongside classification.
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None          # Set to an integer for quick smoke tests.
USE_CPU = False             # Force CPU if needed.

# Hidden sizes for the current classification baselines.
CNN_HIDDEN = 96
GRAPH_HIDDEN = 96
FUSION_HIDDEN = 128
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
MSG_PASSES = 2
DROPOUT = 0.10
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

# Device selection is kept simple and transparent.
device = torch.device('cuda' if torch.cuda.is_available() and not USE_CPU else 'cpu')
print('Device:', device)


Device: cuda


In [4]:
# Shared data and evaluation utilities live in one helper module so that
# all notebooks reuse the same dataset, split, path-remapping, and trajectory metrics.
from dataset_helper import (
    build_run_manifest,
    build_sample_table,
    build_label_mapping,
    compute_trajectory_metrics,
    configure_helper,
    group_streams,
    load_npy_cached,
    prepare_result_dirs,
    resample_scan,
    save_mean_step_error_plot,
    save_or_load_fixed_split,
    save_run_manifest,
    save_training_history,
    save_trajectory_overlay_plots,
    set_seed,
    timestamp_tag,
)

configure_helper(
    dataset_root=DATASET_ROOT,
    original_dataset_root=ORIGINAL_DATASET_ROOT,
    results_root=RESULTS_ROOT,
    weights_root=WEIGHTS_ROOT,
)


In [5]:
@dataclass
class TrajectorySample:
    scan_seq: torch.Tensor | None
    node_seq: torch.Tensor | None
    edge_seq: torch.Tensor | None
    label: torch.Tensor
    future_xy: torch.Tensor
    future_dt: torch.Tensor
    sample_id: str


class BaseTrajectoryDataset(Dataset):
    def __init__(self, streams, sample_table, label_to_idx, past_len, scan_beams, range_clip):
        self.streams = streams
        self.sample_table = sample_table
        self.label_to_idx = label_to_idx
        self.past_len = past_len
        self.scan_beams = scan_beams
        self.range_clip = range_clip

    def __len__(self):
        return len(self.sample_table)

    def _window(self, idx):
        meta = self.sample_table[idx]
        stream = self.streams[meta['stream_idx']]
        return stream[meta['start']: meta['start'] + self.past_len], meta


class ScanOnlyDataset(BaseTrajectoryDataset):
    def __getitem__(self, idx):
        window, meta = self._window(idx)
        scan_seq = []
        for frame in window:
            scan_ref = frame['observation']['ego_planar_scan']
            scan = load_npy_cached(str(scan_ref['path']))
            scan_seq.append(resample_scan(scan, self.scan_beams, self.range_clip))
        label_name = meta.get('raw_label') or 'go_to_goal'
        label_idx = self.label_to_idx.get(label_name, 0)
        return TrajectorySample(
            scan_seq=torch.tensor(np.stack(scan_seq, axis=0), dtype=torch.float32),
            node_seq=None,
            edge_seq=None,
            label=torch.tensor(label_idx, dtype=torch.long),
            future_xy=torch.tensor(meta['future_xy'], dtype=torch.float32),
            future_dt=torch.tensor(meta['future_dt'], dtype=torch.float32),
            sample_id=meta['sample_id'],
        )


def collate_samples(batch):
    scan_seq = None if batch[0].scan_seq is None else torch.stack([sample.scan_seq for sample in batch], dim=0)
    node_seq = None if batch[0].node_seq is None else torch.stack([sample.node_seq for sample in batch], dim=0)
    edge_seq = None if batch[0].edge_seq is None else torch.stack([sample.edge_seq for sample in batch], dim=0)
    labels = torch.stack([sample.label for sample in batch], dim=0)
    future_xy = torch.stack([sample.future_xy for sample in batch], dim=0)
    future_dt = torch.stack([sample.future_dt for sample in batch], dim=0)
    sample_ids = [sample.sample_id for sample in batch]
    return scan_seq, node_seq, edge_seq, labels, future_xy, future_dt, sample_ids


In [6]:
class LidarCNNEncoder(nn.Module):
    """1D CNN encoder for the planar lidar scan."""
    def __init__(self, in_channels=2, hidden_dim=CNN_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, hidden_dim, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


In [7]:
def save_trajectory_history_plot(history: dict, out_path: Path, title_prefix: str):
    if not history or len(history.get('epoch', [])) == 0:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history['epoch'], history['train_loss'], label='train_loss')
    axes[0].plot(history['epoch'], history['val_loss'], label='val_loss')
    axes[0].set_title(f'{title_prefix}: Trajectory Loss')
    axes[0].legend()

    axes[1].plot(history['epoch'], history['val_ade'], label='val_ADE', color='tab:green')
    axes[1].plot(history['epoch'], history['val_fde'], label='val_FDE', color='tab:orange')
    axes[1].plot(history['epoch'], history['val_rmse'], label='val_RMSE', color='tab:red')
    axes[1].set_title(f'{title_prefix}: Validation Trajectory Metrics')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.close(fig)


def evaluate_trajectory_model(model, loader, device, labels):
    model.eval()
    all_targets, all_sample_ids = [], []
    all_pred_future_xy, all_true_future_xy = [], []
    total_loss = 0.0
    total_count = 0
    criterion_traj = nn.MSELoss()
    with torch.no_grad():
        for scan_seq, node_seq, edge_seq, labels_batch, future_xy_batch, _future_dt_batch, sample_ids in loader:
            if scan_seq is not None:
                scan_seq = scan_seq.to(device)
            if node_seq is not None:
                node_seq = node_seq.to(device)
            if edge_seq is not None:
                edge_seq = edge_seq.to(device)
            future_xy_batch = future_xy_batch.to(device)

            pred_future_xy = model(scan_seq, node_seq, edge_seq)
            loss = criterion_traj(pred_future_xy, future_xy_batch)

            total_loss += float(loss.item()) * future_xy_batch.size(0)
            total_count += int(future_xy_batch.size(0))
            all_targets.append(labels_batch.cpu().numpy())
            all_pred_future_xy.append(pred_future_xy.cpu().numpy())
            all_true_future_xy.append(future_xy_batch.cpu().numpy())
            all_sample_ids.extend(sample_ids)

    targets = np.concatenate(all_targets, axis=0)
    pred_future_xy = np.concatenate(all_pred_future_xy, axis=0)
    true_future_xy = np.concatenate(all_true_future_xy, axis=0)

    metrics = compute_trajectory_metrics(pred_future_xy, true_future_xy)
    metrics['loss'] = total_loss / max(total_count, 1)
    return {
        'metrics': metrics,
        'targets': targets,
        'sample_ids': all_sample_ids,
        'pred_future_xy': pred_future_xy,
        'true_future_xy': true_future_xy,
    }


def train_trajectory_model(model, train_loader, val_loader, labels, model_slug, timestamp, split_path, extra_manifest=None, save_weights=True):
    result_dir, weight_dir, plot_dir = prepare_result_dirs(model_slug)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion_traj = nn.MSELoss()

    history = {
        'epoch': [],
        'train_loss': [],
        'val_loss': [],
        'val_ade': [],
        'val_fde': [],
        'val_rmse': [],
    }
    best_val_ade = math.inf
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        running_count = 0
        for scan_seq, node_seq, edge_seq, _labels_batch, future_xy_batch, _future_dt_batch, _sample_ids in train_loader:
            if scan_seq is not None:
                scan_seq = scan_seq.to(device)
            if node_seq is not None:
                node_seq = node_seq.to(device)
            if edge_seq is not None:
                edge_seq = edge_seq.to(device)
            future_xy_batch = future_xy_batch.to(device)

            optimizer.zero_grad()
            pred_future_xy = model(scan_seq, node_seq, edge_seq)
            loss = criterion_traj(pred_future_xy, future_xy_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += float(loss.item()) * future_xy_batch.size(0)
            running_count += int(future_xy_batch.size(0))

        train_loss = running_loss / max(running_count, 1)
        val_eval = evaluate_trajectory_model(model, val_loader, device, labels)
        val_metrics = val_eval['metrics']

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_ade'].append(val_metrics['ADE'])
        history['val_fde'].append(val_metrics['FDE'])
        history['val_rmse'].append(val_metrics['RMSE'])

        print(
            f"Epoch {epoch:02d}/{EPOCHS} "
            f"train_loss={train_loss:.5f} "
            f"val_loss={val_metrics['loss']:.5f} "
            f"val_ADE={val_metrics['ADE']:.4f} "
            f"val_FDE={val_metrics['FDE']:.4f} "
            f"val_RMSE={val_metrics['RMSE']:.4f}"
        )

        if val_metrics['ADE'] < best_val_ade:
            best_val_ade = val_metrics['ADE']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f'Early stopping at epoch {epoch:02d}')
                break

    if best_state is None:
        raise RuntimeError('Training ended without a valid best checkpoint.')

    model.load_state_dict(best_state)
    run_manifest = build_run_manifest(
        model_slug=model_slug,
        timestamp=timestamp,
        labels=labels,
        split_path=split_path,
        extra={**(extra_manifest or {}), 'task': 'trajectory_only', 'selection_metric': 'ADE'},
    )
    manifest_path = save_run_manifest(result_dir, run_manifest, timestamp)

    weight_payload = {
        'model_state': best_state,
        'labels': labels,
        'timestamp': timestamp,
        'model_slug': model_slug,
        'run_manifest': run_manifest,
    }
    if save_weights:
        torch.save(weight_payload, weight_dir / f'{model_slug}_{timestamp}.pt')
        torch.save(weight_payload, weight_dir / 'latest.pt')

    save_training_history(history, result_dir / f'history_{timestamp}.csv')
    save_trajectory_history_plot(history, plot_dir / f'history_{timestamp}.png', model_slug)

    return {
        'history': history,
        'best_val_ade': float(best_val_ade),
        'result_dir': result_dir,
        'plot_dir': plot_dir,
        'weight_dir': weight_dir,
        'run_manifest_path': manifest_path,
        'run_manifest': run_manifest,
    }


def save_final_trajectory_evaluation(model_slug, timestamp, train_out, test_eval, labels, extra_summary=None, split_path=None):
    result_dir = train_out['result_dir'] if train_out is not None else prepare_result_dirs(model_slug)[0]
    plot_dir = train_out['plot_dir'] if train_out is not None else prepare_result_dirs(model_slug)[2]

    metrics = dict(test_eval['metrics'])
    if split_path is not None:
        metrics['split_path'] = str(split_path)
    if train_out is not None and 'best_val_ade' in train_out:
        metrics['best_val_ADE'] = float(train_out['best_val_ade'])
    if extra_summary:
        metrics.update(extra_summary)

    if train_out is not None and 'run_manifest' in train_out:
        (result_dir / 'latest_run_manifest.json').write_text(json.dumps(train_out['run_manifest'], indent=2))

    overlay_paths = save_trajectory_overlay_plots(
        test_eval['pred_future_xy'],
        test_eval['true_future_xy'],
        test_eval['targets'],
        labels,
        plot_dir,
        prefix=timestamp,
        max_plots=8,
    )
    step_error_path = save_mean_step_error_plot(
        test_eval['pred_future_xy'],
        test_eval['true_future_xy'],
        plot_dir / f'mean_step_error_{timestamp}.png',
        title=f'{model_slug} Mean Future-Step Error',
    )
    prediction_export_path = result_dir / f'trajectory_predictions_{timestamp}.npz'
    np.savez_compressed(
        prediction_export_path,
        sample_ids=np.asarray(test_eval['sample_ids'], dtype=str),
        pred_future_xy=test_eval['pred_future_xy'],
        true_future_xy=test_eval['true_future_xy'],
    )
    latest_prediction_export_path = result_dir / 'latest_trajectory_predictions.npz'
    np.savez_compressed(
        latest_prediction_export_path,
        sample_ids=np.asarray(test_eval['sample_ids'], dtype=str),
        pred_future_xy=test_eval['pred_future_xy'],
        true_future_xy=test_eval['true_future_xy'],
    )

    metrics['trajectory_overlay_plots'] = overlay_paths
    metrics['mean_step_error_plot'] = step_error_path
    metrics['trajectory_prediction_file'] = str(latest_prediction_export_path)
    metrics['status'] = 'saved'

    metrics_path = result_dir / f'metrics_{timestamp}.json'
    metrics_path.write_text(json.dumps(metrics, indent=2))
    (result_dir / 'latest_metrics.json').write_text(json.dumps(metrics, indent=2))
    return metrics


In [8]:
set_seed(SEED)
labels, label_mapping = build_label_mapping(LABEL_MODE)
label_to_idx = {label: idx for idx, label in enumerate(labels)}
streams = group_streams(DATASET_ROOT, allowed_labels=None, label_mapping=None)
sample_table = build_sample_table(streams, past_len=PAST_LEN, future_len=FUTURE_LEN)
split_path = SPLITS_ROOT / f'husky_control_split_{LABEL_MODE}_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json'
split_info = save_or_load_fixed_split(sample_table, split_path, SEED, TRAIN_RATIO, VAL_RATIO, PAST_LEN, FUTURE_LEN)
assert split_info['sample_count'] == len(sample_table), (
    f"Split file {split_path} does not match the current dataset: "
    f"split has {split_info['sample_count']} samples, current table has {len(sample_table)}"
)

print('Controller-state metadata labels retained for plots only:', labels)
print('Split path:', split_path)
print('Total samples in canonical table:', len(sample_table))
print('Train / Val / Test:', len(split_info['train_indices']), len(split_info['val_indices']), len(split_info['test_indices']))
print('Future horizon:', split_info['future_len'])


Controller-state metadata labels retained for plots only: ['go_to_goal', 'avoid_left', 'avoid_right', 'commit_forward', 'arrived']
Split path: /home/basudeo/Documents/Thesis/04_model_evaluation/splits/husky_control_split_reduced_seed42_past10_future5.json
Total samples in canonical table: 9012
Train / Val / Test: 6308 1351 1353
Future horizon: 5


In [9]:
class LidarCNNEncoder(nn.Module):
    """1D CNN encoder for the planar lidar scan."""
    def __init__(self, in_channels=2, hidden_dim=CNN_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, hidden_dim, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class CNNLSTMTrajectoryPredictor(nn.Module):
    """Use only ego lidar over time and predict future trajectory."""
    def __init__(self, future_len):
        super().__init__()
        self.future_len = future_len
        self.encoder = LidarCNNEncoder(in_channels=2, hidden_dim=CNN_HIDDEN)
        self.lstm = nn.LSTM(
            input_size=CNN_HIDDEN,
            hidden_size=LSTM_HIDDEN,
            num_layers=LSTM_LAYERS,
            batch_first=True,
            dropout=DROPOUT if LSTM_LAYERS > 1 else 0.0,
        )
        self.traj_head = nn.Sequential(
            nn.Linear(LSTM_HIDDEN, LSTM_HIDDEN),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(LSTM_HIDDEN, future_len * 2),
        )

    def forward(self, scan_seq, node_seq, edge_seq):
        encoded_steps = []
        for t in range(scan_seq.size(1)):
            encoded_steps.append(self.encoder(scan_seq[:, t]))
        seq = torch.stack(encoded_steps, dim=1)
        _, (h, _c) = self.lstm(seq)
        hidden = h[-1]
        future_xy = self.traj_head(hidden).view(hidden.size(0), self.future_len, 2)
        return future_xy


In [10]:
# Build the dataset for this model family.
DATASET_CLASS = ScanOnlyDataset
MODEL_SLUG = 'cnn_lstm'
TIMESTAMP = timestamp_tag()

dataset = DATASET_CLASS(
    streams=streams,
    sample_table=sample_table,
    label_to_idx=label_to_idx,
    past_len=PAST_LEN,
    scan_beams=SCAN_BEAMS,
    range_clip=RANGE_CLIP,
)

train_subset = Subset(dataset, split_info['train_indices'])
val_subset = Subset(dataset, split_info['val_indices'])
test_subset = Subset(dataset, split_info['test_indices'])

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_samples)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_samples)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_samples)

MODEL_EXTRA_MANIFEST = {
    'cnn_hidden': CNN_HIDDEN,
    'lstm_hidden': LSTM_HIDDEN,
    'lstm_layers': LSTM_LAYERS,
    'dropout': DROPOUT,
    'future_len': FUTURE_LEN,
}
model = CNNLSTMTrajectoryPredictor(future_len=FUTURE_LEN).to(device)
train_out = train_trajectory_model(
    model,
    train_loader,
    val_loader,
    labels,
    MODEL_SLUG,
    TIMESTAMP,
    split_path=split_path,
    extra_manifest=MODEL_EXTRA_MANIFEST,
    save_weights=True,
)
test_eval = evaluate_trajectory_model(model, test_loader, device, labels)
final_metrics = save_final_trajectory_evaluation(MODEL_SLUG, TIMESTAMP, train_out, test_eval, labels, split_path=split_path)
print('Final trajectory metrics:')
print(json.dumps(final_metrics, indent=2))


Epoch 01/30 train_loss=1648.20197 val_loss=1626.37632 val_ADE=16.9738 val_FDE=27.2355 val_RMSE=57.0329
Epoch 02/30 train_loss=1648.18824 val_loss=1626.36738 val_ADE=16.9813 val_FDE=27.2357 val_RMSE=57.0328
Epoch 03/30 train_loss=1648.24067 val_loss=1626.36539 val_ADE=16.9926 val_FDE=27.2443 val_RMSE=57.0327
Epoch 04/30 train_loss=1648.10616 val_loss=1626.37131 val_ADE=17.0064 val_FDE=27.2544 val_RMSE=57.0328
Epoch 05/30 train_loss=1648.16164 val_loss=1626.37222 val_ADE=17.0124 val_FDE=27.2624 val_RMSE=57.0328
Epoch 06/30 train_loss=1648.08493 val_loss=1626.36666 val_ADE=17.0098 val_FDE=27.2663 val_RMSE=57.0327
Early stopping at epoch 06
Final trajectory metrics:
{
  "ADE": 16.90874481201172,
  "FDE": 26.110857009887695,
  "RMSE": 56.97283935546875,
  "loss": 1622.9522886875375,
  "split_path": "/home/basudeo/Documents/Thesis/04_model_evaluation/splits/husky_control_split_reduced_seed42_past10_future5.json",
  "best_val_ADE": 16.97377586364746,
  "trajectory_overlay_plots": [
    "/home